# V11 – Programmieren mit Copilot – Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt im Labor, dann das Training zu Hause (Browser, ca. 60 Min). Dieses Notebook ist die **Vertiefung** und läuft am besten im **Codespace** (Anleitung: Lerneinheit V11, Abschnitt 3). Dort hast du Copilot, ein Terminal und die Möglichkeit, eigene Dateien anzulegen. Es nutzt Python aus V01 bis V11.

**Der Arbeitsablauf bei jeder Aufgabe in Stufe 2 und 3:** (1) Auftrag als PAP und Text festhalten, (2) Testfälle **von Hand** in die Zelle schreiben, (3) erst dann Copilot fragen, (4) Code **lesen** und mit den Testfällen laufen lassen, (5) freigeben oder zurück zu (1). Die Selbsttests sind keine Musterlösungen: Sie zeigen nur, ob dein Ergebnis stimmt.

**Kleines Vokabular:** Ein **Modul** ist eine Python-Datei mit fertigen Funktionen, die du mit `import` holst. Die **Standardbibliothek** ist die Sammlung, die mit jedem Python kommt (`math`, `statistics`, `random`). `pip` installiert **Pakete** aus dem Paketindex PyPI. Ein **Prompt** ist die Anweisung an die KI. Ein **Grenzfall** ist ein Eingabewert genau auf der Grenze (zum Beispiel Spiel genau 0).

**Datenschutz:** Keine Firmen-, Kunden- oder Zugangsdaten in Prompts. Hier genügen Übungsdaten.

In [ ]:
# Werkzeug-Zelle für die Selbsttests: einfach einmal ausführen, nicht verändern.
# (Wie das innen funktioniert, lernst du in V07 und V12. Du musst es nicht verstehen.)
def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

def pruefe_nah(ist, soll, text, toleranz=1e-9):
    # Für Kommazahlen: "gleich" heißt hier, die Abweichung ist winzig (Rundungsfehler!).
    if isinstance(ist, (int, float)) and abs(ist - soll) <= toleranz:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

## Stufe 1 – Aufwärmen

### 1.1 Vorhersagen: Welche Namen gibt es?
Schreibe **vor dem Ausführen** als Kommentar in die erste Zeile, welche der Zeilen einen Fehler auslösen und welchen. Prüfe dann nach. Lies die Meldung von unten nach oben.

In [ ]:
# Meine Vorhersage: ...
import math as m
print(m.sqrt(49))
# print(math.sqrt(49))     # Entferne das # und probiere es aus
# print(sqrt(49))          # dito

### 1.2 Selbst bauen: drei Schreibweisen
Schreibe die `import`-Zeilen selbst. (a) `import math`: Umfang eines Kreises mit Radius 12,5 mm. (b) `from statistics import mean`: Mittelwert der Messwerte. (c) `import random as zufall`: eine ganze Zufallszahl von 1 bis 6. Ersetze die `None`.

In [ ]:
# TODO: die drei import-Zeilen hier
radius = 12.5
messwerte = [24.98, 25.02, 25.00, 25.01]
umfang = None    # 2 * pi * radius
mittel = None    # Mittelwert
wurf = None      # ganze Zahl 1 bis 6
print(umfang, mittel, wurf)

In [ ]:
pruefe_nah(umfang, 78.53981633974483, "Umfang")
pruefe_nah(mittel, 25.0025, "Mittelwert")
pruefe(isinstance(wurf, int) and 1 <= wurf <= 6, True, "Wurf von 1 bis 6")

### 1.3 Selbst bauen: Median und Standardabweichung
Berechne für die Messreihe mit dem Modul `statistics` den Median und die Stichproben-Standardabweichung (`stdev`, geteilt durch n − 1). Vergleiche den Median mit dem Mittelwert: Was fällt auf, wenn ein Ausreißer dabei ist?

In [ ]:
import statistics

werte = [12.004, 12.011, 11.998, 12.007, 12.019, 12.600]    # der letzte Wert ist ein Ausreißer
median = None    # TODO
mittel = None    # TODO
streuung = None  # TODO: stdev
print(median, mittel, streuung)

In [ ]:
pruefe_nah(median, 12.009, "Median")
pruefe_nah(mittel, 12.1065, "Mittelwert (vom Ausreißer nach oben gezogen)", 1e-9)
pruefe(round(streuung, 3) if streuung is not None else None, 0.242, "Standardabweichung gerundet")

### 1.4 Ein eigenes Modul anlegen und benutzen
Mit `%%writefile` schreibt die Zelle ihren Inhalt in eine Datei im Ordner des Notebooks. Danach kannst du sie importieren. (Wenn du die Datei änderst, starte den Kernel neu oder benutze `importlib.reload`.)

In [ ]:
%%writefile passungen.py
def passungsart(bohrung_min, bohrung_max, welle_min, welle_max):
    """Liefert 'Spielpassung', 'Übermaßpassung' oder 'Übergangspassung' (Maße in mm)."""
    if bohrung_min >= welle_max:
        return "Spielpassung"
    elif bohrung_max <= welle_min:
        return "Übermaßpassung"
    else:
        return "Übergangspassung"

In [ ]:
import passungen

art = None    # TODO: Passungsart für 25 H7/k6 (25.000, 25.021, 25.002, 25.015) mit dem Modul passungen
print(art)

In [ ]:
pruefe(art, "Übergangspassung", "H7/k6 ist eine Übergangspassung")

## Stufe 2 – Anwenden

### 2.1 Testfälle zuerst: eine Auslastungs-Ampel
Bei einem Bauteil ist die Auslastung `a = f_ist / f_zul` (Kräfte in N). Bei `a` bis einschließlich 0,8 zeigt die Ampel `"gruen"`, bis einschließlich 1,0 `"gelb"`, darüber `"rot"`.

**Schritt 1 (ohne KI):** Schreibe mindestens fünf Testfälle als Liste von Tupeln `(f_ist, f_zul, erwartet)`, darunter die beiden Grenzen (a genau 0,8 und a genau 1,0).

In [ ]:
testfaelle = [
    (40, 100, "gruen"),
    # TODO: weitere Fälle, auch die Grenzen
]
print(len(testfaelle), "Testfälle")

**Schritt 2:** Schreibe den Prompt im Copilot-Chat (Eingaben, Ausgaben, Einheiten, Grenzfälle, Beispiel) und lass dir die Funktion `auslastung_ampel(f_ist, f_zul)` erzeugen. Füge sie in die nächste Zelle ein und **lies** sie, bevor du weitermachst. Ohne Copilot schreibst du sie selbst.

In [ ]:
# Hier die Funktion einfügen
def auslastung_ampel(f_ist, f_zul):
    return None

In [ ]:
# Testschleife: Deine Testfälle gegen die Funktion
fehler = 0
for f_ist, f_zul, erwartet in testfaelle:
    ergebnis = auslastung_ampel(f_ist, f_zul)
    if ergebnis == erwartet:
        print("ok    ", f_ist, f_zul, ergebnis)
    else:
        print("FEHLER", f_ist, f_zul, "->", ergebnis, "erwartet", erwartet)
        fehler = fehler + 1
print("Fehlgeschlagene Tests:", fehler)

In [ ]:
pruefe(len(testfaelle) >= 5, True, "mindestens fünf Testfälle")
pruefe(auslastung_ampel(80, 100), "gruen", "Grenze 0,8 ist noch gruen")
pruefe(auslastung_ampel(100, 100), "gelb", "Grenze 1,0 ist noch gelb")
pruefe(auslastung_ampel(120, 100), "rot", "über 1,0 ist rot")

### 2.2 KI-Code prüfen: drei eingebaute Fehler
Diesen Code hat eine KI für „Passungsart und Prüfung des mittleren Höchstspiels im Soll 0,022 mm (Toleranz 0,005 mm)“ geliefert. Er enthält drei Fehler, die du aus dem Labor kennst (Grenze, erfundene Funktion, erfundener Import). Finde sie, **ohne** den Code als Ganzes zu starten, und repariere ihn in der nächsten Zelle. Teste die Reparatur mit den Selbsttests.

In [ ]:
# KI-Antwort (Fehler eingebaut), nicht ausführen
# import math
# import statistics
# import passungstools
#
# def passungsart(bohrung_min, bohrung_max, welle_min, welle_max):
#     if bohrung_min > welle_max:
#         return "Spielpassung"
#     elif bohrung_max <= welle_min:
#         return "Übermaßpassung"
#     else:
#         return "Übergangspassung"
#
# hoechstspiele = [0.041, -0.007, 0.019, 0.034]
# mittel = statistics.mean(hoechstspiele)
# im_soll = math.tolerance(mittel, 0.022, 0.005)

In [ ]:
# Deine korrigierte Fassung:
import statistics
# TODO: Import für die Toleranzprüfung

def passungsart_ki(bohrung_min, bohrung_max, welle_min, welle_max):
    pass    # TODO

hoechstspiele = [0.041, -0.007, 0.019, 0.034]
mittel = statistics.mean(hoechstspiele)
im_soll = None    # TODO: Toleranzprüfung mit isclose
print(mittel, im_soll)

In [ ]:
pruefe(passungsart_ki(25.000, 25.021, 24.987, 25.000), "Spielpassung", "Grenzfall Spiel genau 0")
pruefe(passungsart_ki(25.000, 25.021, 25.021, 25.040), "Übermaßpassung", "Grenzfall Übermaß genau 0")
pruefe(passungsart_ki(25.000, 25.021, 25.002, 25.015), "Übergangspassung", "Übergang")
pruefe(im_soll, True, "Mittel 0,02175 liegt im Soll 0,022 +- 0,005")

### 2.3 Prompt verbessern
Der vage Prompt: *„Mach mir was für die Festigkeit.“* Gemeint war: `sicherheit(kraft, querschnitt, streckgrenze)` mit Kraft in N, Querschnitt in mm², Streckgrenze in MPa, Sicherheit S = Re / σ mit σ = F / A, „ausreichend“ ab S = 1,5 (genau 1,5 zählt dazu). Schreibe den Prompt in die Zelle, schick ihn an Copilot und notiere unten, was Copilot **zusätzlich geraten** hat (Zahlenwerte, Grenzen, Rückgabetexte).

In [ ]:
prompt = """
"""
raten = """
Notizen: Was hat die KI geraten, obwohl ich es nicht gesagt hatte?
"""
print(len(prompt.split()), "Wörter im Prompt")

In [ ]:
import re
kriterien = {
    "Eingaben": r"\w+\(\s*\w+(?:\s*,\s*\w+)+\s*\)",
    "Ausgaben": r"rückgabe|zurück|liefert|gibt|return",
    "Einheiten": r"\bN\b.*mm|mm.*\bN\b",
    "Grenzfälle": r"genau|gleich|>=|grenzfall",
    "Beispiele": r"beispiel|->|erwartet",
}
for name, muster in kriterien.items():
    print(("OK:     " if re.search(muster, prompt, re.I) else "FEHLT:  ") + name)
# Diese Stichwortsuche ersetzt kein Nachdenken: Sie erkennt nur, ob die Angabe erwähnt ist.

## Stufe 3 – Knobeln

### 3.1 Prompt A gegen Prompt B
Stelle derselben KI zwei Prompts für „Passungsart bestimmen“: A als ein Satz, B als PAP in Textform mit Grenzfall und Beispiel. Füge beide Antworten ein und teste sie mit **denselben** Testfällen aus der Lerneinheit (g6, r6, k6 und h6 als Grenzfall). Schreibe auf: Welche Antwort besteht welche Tests, und was hat Prompt A geraten?

In [ ]:
testfaelle_passung = [
    (25.000, 25.021, 24.980, 24.993, "Spielpassung"),
    (25.000, 25.021, 25.028, 25.041, "Übermaßpassung"),
    (25.000, 25.021, 25.002, 25.015, "Übergangspassung"),
    (25.000, 25.021, 24.987, 25.000, "Spielpassung"),     # Grenzfall
]

def antwort_a(bohrung_min, bohrung_max, welle_min, welle_max):
    return None    # TODO: Funktion aus der Antwort auf Prompt A (angepasst auf diese Signatur)

def antwort_b(bohrung_min, bohrung_max, welle_min, welle_max):
    return None    # TODO: Funktion aus der Antwort auf Prompt B

for name, f in (("A", antwort_a), ("B", antwort_b)):
    bestanden = 0
    for a, b, c, d, erwartet in testfaelle_passung:
        if f(a, b, c, d) == erwartet:
            bestanden = bestanden + 1
    print("Antwort", name, ":", bestanden, "von", len(testfaelle_passung), "Tests bestanden")

### 3.2 Gibt es die Funktion wirklich?
Bevor du eine Funktion aus KI-Code benutzt, kannst du nachsehen, ob das Modul sie hat. `dir(modul)` listet alle Namen eines Moduls, und `name in dir(modul)` fragt, ob ein Name dabei ist. Prüfe, welche dieser KI-Funktionen es wirklich gibt, und trage `True` oder `False` ein.

In [ ]:
import math
import statistics

gibt_es = {
    "math.sqrt": None,         # TODO: True oder False
    "math.tolerance": None,
    "math.isclose": None,
    "statistics.median": None,
    "statistics.mittelwert": None,
}
for name in gibt_es:
    print(name, gibt_es[name])

In [ ]:
pruefe(gibt_es["math.sqrt"], "sqrt" in dir(math), "math.sqrt")
pruefe(gibt_es["math.tolerance"], "tolerance" in dir(math), "math.tolerance")
pruefe(gibt_es["math.isclose"], "isclose" in dir(math), "math.isclose")
pruefe(gibt_es["statistics.median"], "median" in dir(statistics), "statistics.median")
pruefe(gibt_es["statistics.mittelwert"], "mittelwert" in dir(statistics), "statistics.mittelwert")

### 3.3 pip im Terminal
Öffne im Codespace ein Terminal (nicht diese Zelle) und probiere nacheinander aus: `pip list` (installierte Pakete), `pip show numpy` (Version und Ort) und `pip install requests==2.32.3` in einer **virtuellen Umgebung** (`python -m venv .venv`, dann aktivieren). Schreibe auf, was `pip list` vorher und nachher zeigt und welche Version `requests` hat. Installiere nichts, dessen Namen dir eine KI vorgeschlagen hat, ohne vorher auf PyPI nachzusehen, ob es das Paket gibt, wie alt es ist und wer es pflegt.

In [ ]:
meine_notizen = """
pip list (vorher):
pip list (nachher):
Version von requests:
Wie prüfe ich ein Paket, bevor ich es installiere?
"""
print(meine_notizen)

### 3.4 Zum Abschluss: Eigene Spezifikation
Wähle ein kleines Ingenieurwerkzeug (zum Beispiel Drehzahl aus Schnittgeschwindigkeit, Spannung aus Kraft, Umrechnung Grad-Bogenmaß). Schreibe (1) den PAP als Skizze oder Text, (2) fünf Testfälle mit Grenzen, (3) den Prompt, (4) lass Copilot die Funktion schreiben und (5) prüfe. Notiere, wo Copilot geraten hat und wie du es gemerkt hast. Diese Reihenfolge ist die Arbeitsweise, die du ab V14 im Projekt brauchst.